# Title

Description

## Imports
Description

In [3]:
import csv
import json
import re
import typing
from pathlib import Path

import numpy as np
import pandas as pd
from charset_normalizer import from_path
#

## Task 1
Description

Description

## Task 2
Description

Description

## Task 3
Description

We want to remove glitch ouliers in the prices, as wall as turn the prices into a 1D array. We also want to calculate the refund ratio using the refund count and the sales count. 

In [ ]:


def remove_price_outliers(prices):
    ''' 
    Converts a list of prices to a 1D numpy array and removes outliers from a list of prices using 3 standard deviations from the mean. The outliers are replaced 
    with the median of the non-outlier prices.The function returns a 1D array.
    '''

    data = np.asarray(prices, dtype=float).flatten()

    mean = np.mean(data)
    std = np.std(data)
    outliers = np.abs(data - mean) > 3 * std

    median = np.median(data[~outliers])
    data[outliers] = median

    return data


def compute_refund_ratio(sales_count: np.ndarray, refund_count: np.ndarray) -> np.ndarray:
    '''
    Computes the refund ratio given the sales count and refund count. The addition of a small epsilon value is added to avoid zero-division errors. The function 
    returns a 1D array.
    '''
    
    epsilon = 1e-6
    refund_ratio = refund_count / (sales_count + epsilon)

    return refund_ratio



Description

## Task 4
Description

The transactions are converted into a pandas DataFrame, with time as the index, and sorts them cronologically. We group by item and resampple into 1 hour windows to sum the price, build a pivot table of revenue per item across success, refund and cancelled. We also compute the refund ratio per item by using the function form task 3.

In [ ]:


def create_dataframe(transactions):
    '''
    Creates a pandas DataFrame from a list of transactions. The function converts the
    'timestamp' column to a datetime object and sets it as the index of the DataFrame.
    '''

    df = pd.DataFrame(transactions)
    df['time'] = pd.to_datetime(df['timestamp'])
    df = df.set_index('timestamp')
    df = df.sort_index()

    return df


def hourly_revenue(df):
    '''
    Computes the hourly revenue for each item in the DataFrame. The function groups the
    DataFrame by 'item' and resamples the 'price' column to hourly intervals,
    '''
    return df.groupby('item').resample('1h')['price'].sum()


def status_pivot(df):
    ''' 
    Creates a pivot table from the DataFrame with 'item' as the index, 'status' as the 
    columns, and the sum of 'price' as the values. The function fills any missing values with 0.
    '''
    return pd.pivot_table(
        df,
        values='price',
        index='item',
        columns='status',
        aggfunc='sum',
        fill_value=0
    )


def refund_ratio_per_item(df):
    '''
    Computes the refund ratio for each item in the DataFrame. The function first creates a pivot table 
    using the `status_pivot` function, then extracts the sales and refund counts for each item. Finally, 
    it computes the refund ratio using the `compute_refund_ratio` function and returns a 1D array of refund ratios.
    '''
    pivot = status_pivot(df)
    sales = pivot['SUCCESS']
    refunds = pivot['REFUND']
    
    return compute_refund_ratio(
        np.asarray(sales.values), np.asarray(refunds.values)
    )



Description

## Task 5
Description

Description